In [16]:
from pyspark.sql import functions as f
from pyspark.sql.functions import *
from pyspark.sql import Window
from datetime import datetime

In [17]:
bi_carteira_cobranca = spark.read.parquet('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/BI_CARTEIRA_COBRANCA/')

In [15]:
bi_contrato = spark.read.parquet ('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/BI_CONTRATO/')

base_zeus = spark.read.parquet ('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/BASE_ZEUS/')

titulos_vertia = spark.read.parquet ('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/BI_TITULOS_VERTIA/')

bi_originais = spark.read.parquet ('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/BI_ORIGINAIS/')

bi_acordos_promessas = spark.read.parquet ('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/BI_ACORDOS_PROMESSAS/')

polos_unidades = spark.read.parquet('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/POLOS_PARCEIROS_E_PROPRIOS/')

bi_recebimento = spark.read.parquet('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/BI_RECEBIMENTOS/')

In [1]:
df_sit_ead = spark.read.parquet('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/ALUNSITU_EAD/')
df_sit_pre = spark.read.parquet('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/ALUNSITU_PRESENCIAL/')

17914653 [Thread-5] WARN  org.apache.hadoop.metrics2.impl.MetricsConfig  - Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [5]:
df_sit_ead = df_sit_ead.select(f.col('COD_UNDE_ALNO_ESPC'),f.col('DAT_STCO_ACDM'),f.col('DAT_STCO_ACDM'))
df_sit_pre = df_sit_pre.select(f.col('COD_UNDE_ALNO_ESPC'),f.col('DAT_STCO_ACDM'),f.col('DAT_STCO_ACDM'))

In [13]:
merged_df_sit = df_sit_ead.union(df_sit_pre)

In [11]:
merged_df.count()

18721909

In [18]:
bi_carteira_cobranca.printSchema()

root
 |-- NOM_AGENCIA_EXTERNA: string (nullable = true)
 |-- DAT_PROCESSAMENTO: date (nullable = true)
 |-- CLS_MARCA_CONTRATO: string (nullable = true)
 |-- COD_FILIAL: string (nullable = true)
 |-- DSC_FILIAL: string (nullable = true)
 |-- CLS_SITUACAO_CONTRATO_ALUNO: string (nullable = true)
 |-- DSC_SITUACAO_CONTRATO: string (nullable = true)
 |-- COD_CLIENTE: string (nullable = true)
 |-- COD_CONTRATO: string (nullable = true)
 |-- COD_CONTRATO_ORIGEM: string (nullable = true)
 |-- COD_CPFCGC_PES: string (nullable = true)
 |-- NOM_CLIENTE: string (nullable = true)
 |-- CLS_SCORE_COLLECTION: string (nullable = true)
 |-- CLS_CLASSE_DEBITO: string (nullable = true)
 |-- CLS_CLUSTER: string (nullable = true)
 |-- CLS_SCORE_GLOBAL: string (nullable = true)
 |-- CLS_SCORE_ENGAJAMENTO: string (nullable = true)
 |-- DAT_INICIO_CURSO: string (nullable = true)
 |-- DAT_FIM_CURSO: string (nullable = true)
 |-- DAT_INICIO_MATRICULA: string (nullable = true)
 |-- DAT_FIM_MATRICULA: string (nu

In [ ]:
bi_acordos_promessas = bi_acordos_promessas.select(f.col('cod_parcelamento'),f.col(''),f.col('DAT_CRIACAO_PARCELAMENTO'),f.col('COD_CONTRATO'))

In [193]:
def join_recebimento(bi_recebimento, bi_acordos_promessas):
    #bi_acordos_promessas = bi_acordos_promessas.filter(f.col("NUM_QTDE_PARCELAS") == 1 )
    
    df_bi_recebimento = (bi_recebimento.join(bi_acordos_promessas, 
                           bi_recebimento['cod_parcelamento'] == bi_acordos_promessas['cod_parcelamento'], "left"))
    return df_bi_recebimento.drop(bi_acordos_promessas['cod_parcelamento']).drop(bi_acordos_promessas['COD_CONTRATO'])

In [194]:
def join_carteira_cobranca(df_bi_recebimento, bi_carteira_cobranca):
    df_bi_recebimento = (df_bi_recebimento.join(bi_carteira_cobranca, 
                           df_bi_recebimento['COD_CONTRATO'] == bi_carteira_cobranca['COD_CONTRATO'], "left"))
    return df_bi_recebimento.drop(bi_carteira_cobranca['COD_CONTRATO']).drop(bi_carteira_cobranca['TIP_MATRICULA'])

In [195]:
def join_base_zeus(df_bi_recebimento, base_zeus):
    df_bi_recebimento =  (df_bi_recebimento.withColumn("COD_CPFCGC_PES", f.when(f.col("TIP_MATRICULA") == "PRESENCIAL",
                        f.concat(
                        f.lpad(f.col("COD_FILIAL"),3,'0'), f.lit(""), f.substring(f.col('COD_CONTRATO_ORIGEM'), 2, 15)))
                         .otherwise(f.concat(f.lit("EAD"),f.lit(""), f.col("COD_CONTRATO_ORIGEM")))))
    df_bi_recebimento = (df_bi_recebimento.join(base_zeus, 
                       df_bi_recebimento['COD_CPFCGC_PES'] == base_zeus['COD_RA_ZEUS'], "left"))
    return df_bi_recebimento.drop(base_zeus['COD_RA_ZEUS'])  

In [196]:
def join_titulos_vertia(df_bi_recebimento, titulos_vertia):
    df_bi_recebimento = (df_bi_recebimento.join(titulos_vertia, 
                           f.ltrim(f.rtrim(df_bi_recebimento['COD_CPFCGC_PES'])) == f.ltrim(f.rtrim(titulos_vertia['COD_CPFCGC_PES'])), "left"))
    return df_bi_recebimento.drop(titulos_vertia['COD_CPFCGC_PES']).drop(titulos_vertia["COD_CONTRATO"])

In [197]:
def join_originais(df_bi_recebimento, bi_originais):    
    df_bi_recebimento = (df_bi_recebimento.join(bi_originais, 
                           (df_bi_recebimento['COD_PARCELAMENTO'] == bi_originais['COD_PARCELAMENTO_ORI']) &
                           (bi_originais['NOM_NATUREZA_TITULO'] == "MENSALIDADE") &
                           (bi_originais['IND_NATUREZA_REMATRICULA'] == "S"), "left"))
    return df_bi_recebimento.drop(bi_originais['COD_PARCELAMENTO_ORI'])

In [198]:
def join_bi_contrato(df_bi_recebimento, bi_contrato):
    df_bi_recebimento = (df_bi_recebimento.join(bi_contrato, 
                           df_bi_recebimento['COD_CONTRATO'] == bi_contrato['COD_CONTRATO'], "left"))
    return df_bi_recebimento.drop(bi_contrato['COD_CONTRATO'])

In [199]:
def join_polos_unidades(df_bi_recebimento, polos_unidades):
    df_bi_recebimento = df_bi_recebimento.drop(bi_carteira_cobranca["COD_FILIAL"])
    df_bi_recebimento = (df_bi_recebimento.join(polos_unidades, 
                           (df_bi_recebimento['COD_FILIAL'] == polos_unidades['COD_CLBR']) &
                           (df_bi_recebimento['COD_FILIAL'] == polos_unidades['COD_OLMP']), "left"))
    
    return df_bi_recebimento.drop(polos_unidades['COD_OLMP']).drop(polos_unidades['COD_CLBR'])

In [200]:
df_bi_recebimento = join_recebimento(bi_recebimento, bi_acordos_promessas)

In [ ]:
df_bi_recebimento = join_carteira_cobranca(df_bi_recebimento, bi_carteira_cobranca)

In [182]:
df_bi_recebimento = join_base_zeus(df_bi_recebimento, base_zeus)

In [183]:
df_bi_recebimento = join_titulos_vertia(df_bi_recebimento, titulos_vertia)

In [184]:
df_bi_recebimento = join_originais(df_bi_recebimento, bi_originais)

In [185]:
df_bi_recebimento = join_bi_contrato(df_bi_recebimento, bi_contrato)

In [186]:
df_bi_recebimento = join_polos_unidades(df_bi_recebimento, polos_unidades)

In [187]:
    df_bi_recebimento = (df_bi_recebimento
    .drop(bi_contrato["NOM_MARCA_CONTRATO"])
    .drop(bi_originais["STA_TITULO"])
    .drop(titulos_vertia["STA_TITULO"])
    .drop(bi_contrato["TIP_MATRICULA"])
    .drop(titulos_vertia["DAT_PAGAMENTO"])
    )
df_bi_recebimento = (df_bi_recebimento.filter(f.col("STA_TITULO") == "PAGO"))    


df_bi_recebimento = (df_bi_recebimento
                        .withColumn("DSC_CANAL_REPORT",
                        f.when(f.ltrim(f.rtrim(f.col("NOM_USUARIO_ACORDO"))) == "raid-portal-aluno", "raid-portal-aluno")
                        .when((f.col("NOM_GRUPO_USUARIO").isNull()) & (f.ltrim(f.rtrim(f.col("NOM_USUARIO_ACORDO"))) == "adm"), "adm")
                        .otherwise(
                            f.col("NOM_USUARIO_ACORDO")
                    )))

df_bi_recebimento = (df_bi_recebimento
                        .withColumn("NOM_CARTEIRA",
                        f.when(("TIP_MATRICULA" == f.lit("TIP_MATRICULA")) & (f.col("NOM_MARCA_CONTRATO").isin('ANHANGUERA','UNIDERP')), "AEDU PRESENCIAL")
                        .when(("TIP_MATRICULA" == f.lit("EAD")) & (f.col("NOM_MARCA_CONTRATO").isin('ANHANGUERA','UNIDERP')), "AEDU EAD")
                        .when(("TIP_MATRICULA" == f.lit("PRESENCIAL")) & (f.col("NOM_MARCA_CONTRATO").isin('FAMA','PITAGORAS','UNIC','UNIME','UNOPAR','UNIRONDON')), "DM PRESENCIAL")
                        .when(("TIP_MATRICULA" == f.lit("EAD")) & (f.col("TIP_MATRICULA").isin('UNIME','UNOPAR','PITAGORAS','UNIASSELVI','PITAGORAS','FAMA-MACAPA','UNIC','UNIRONDON')), "UNOPAR EAD")
                    ))

df_bi_recebimento = (df_bi_recebimento.withColumn("NOM_CANAL_COMISSAO", f.lit("RAID_ATP_INTERVALOR")))

df_bi_recebimento = (df_bi_recebimento.filter(
                      (f.lower(f.col("NOM_USUARIO_ACORDO")).like("%-mal%")) &
                      (f.lower(f.col("NOM_GRUPO_USUARIO")).like("%atp_malta%")) &
                      (f.col("DAT_PAGAMENTO") >= '2021-09-10')
                    ))

df_bi_recebimento = (df_bi_recebimento.withColumn("CARTEIRA", f.lit("VERTIA")))

df_bi_recebimento = (df_bi_recebimento.withColumn("NATUREZA_TITULO", f.lit("MENSALIDADE")))
df_bi_recebimento = (df_bi_recebimento.withColumn("NATUREZA_REMATRICULA", f.lit("S")))

df_bi_recebimento = df_bi_recebimento.filter(
                                              (f.col("NUM_DIAS_ATRASO").isNull())|
                                              (f.col("NUM_DIAS_ATRASO") == 0)
                                            )

df_bi_recebimento = (df_bi_recebimento.select(
    df_bi_recebimento["NOM_CARTEIRA"],
    df_bi_recebimento["DSC_CANAL_REPORT"],
    bi_acordos_promessas["STA_PARCELAMENTO"],
    bi_acordos_promessas["VLR_PRINCIPAL"],
    bi_acordos_promessas["VLR_DESC_PRINCIPAL"],
    bi_acordos_promessas["VLR_MULTA"],
    bi_acordos_promessas["VLR_DESC_MULTA"],
    bi_acordos_promessas["VLR_JUROS_MORA"],
    bi_acordos_promessas["VLR_JUROS_FINANC"],
    bi_acordos_promessas["VLR_DESC_JUROS_FINANC"],
    bi_acordos_promessas["VLR_CORRECAO_MONETARIA"],
    bi_acordos_promessas["VLR_DESC_CORRECAO_MONETARIA"],
    base_zeus["TIP_CURSO_ZEUS"],
    bi_acordos_promessas["VLR_DESC_TAXA_NEGOCIACAO"],
    bi_acordos_promessas["VLR_TAXA_NEGOCIACAO"],
    bi_acordos_promessas["NUM_PAGAMENTO_ENTRADA"],
    bi_recebimento["VLR_PARCELA"],
    df_bi_recebimento["NUM_DIAS_ATRASO"],
    bi_acordos_promessas["COD_CLIENTE"],
    bi_acordos_promessas["COD_CONTATO"],
    df_bi_recebimento["COD_PARCELAMENTO"],
    df_bi_recebimento["TIP_MATRICULA"],
    bi_recebimento["NOM_PRODUTO_PEP"],
    bi_originais["COD_TITULO"],
    bi_recebimento["NOM_GRUPO_USUARIO"],
    bi_recebimento["NOM_USUARIO_ACORDO"],
    bi_originais["DAT_VENCIMENTO"],
    bi_originais["NUM_PARCELA"],
    bi_recebimento["DAT_PAGAMENTO"],
    bi_recebimento["VLR_PAGAMENTO"],
    bi_acordos_promessas["DSC_NATUREZA_REMATRICULA"],
    bi_recebimento["TIP_NATUREZA_TITULO"],
    bi_recebimento["DSC_NATUREZA_TITULO"],
    bi_originais["NUM_MES_COMPETENCIA"],
    bi_originais["NUM_ANO_COMPETENCIA"]
    ))


8906597 [Thread-5] WARN  org.apache.spark.sql.Column  - Constructing trivially true equals predicate, 'TIP_MATRICULA = TIP_MATRICULA'. Perhaps you need to use aliases.


In [188]:
df_bi_recebimento.show()

+---------------+--------------------+----------------+-------------+------------------+---------+--------------+--------------+----------------+---------------------+----------------------+---------------------------+--------------+------------------------+-------------------+---------------------+-----------+---------------+-----------+-----------+----------------+-------------+---------------+----------+-----------------+--------------------+--------------+-----------+--------------------+-------------+------------------------+-------------------+-------------------+-------------------+-------------------+
|   NOM_CARTEIRA|    DSC_CANAL_REPORT|STA_PARCELAMENTO|VLR_PRINCIPAL|VLR_DESC_PRINCIPAL|VLR_MULTA|VLR_DESC_MULTA|VLR_JUROS_MORA|VLR_JUROS_FINANC|VLR_DESC_JUROS_FINANC|VLR_CORRECAO_MONETARIA|VLR_DESC_CORRECAO_MONETARIA|TIP_CURSO_ZEUS|VLR_DESC_TAXA_NEGOCIACAO|VLR_TAXA_NEGOCIACAO|NUM_PAGAMENTO_ENTRADA|VLR_PARCELA|NUM_DIAS_ATRASO|COD_CLIENTE|COD_CONTATO|COD_PARCELAMENTO|TIP_MATRICULA|

In [189]:
df_bi_recebimento.count()

398